<a href="https://colab.research.google.com/github/asmipandey18/Elevvo_Ml_Internship/blob/main/task9_predictive_maintenance/Predictive_maintenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 9: Industrial Predictive Maintenance

**Goal:** Predict machine failures from sensor readings, while keeping false alarms low. In a factory, a false alarm stops production for nothing, so the model is judged mainly by its False Discovery Rate (false alarms ÷ all alarms) and not by accuracy alone.

**Dataset:** AI4I 2020 Predictive Maintenance Dataset (Kaggle, original upload by Matzka), 10,000 machine records with air temperature, process temperature, rotational speed, torque and tool wear.

**Tools:** Python, Pandas, Matplotlib, Seaborn, Scikit-learn, XGBoost

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/content/ai4i2020.csv")

In [3]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
df.info()
print(df.columns.tolist())
print(df["Machine failure"].value_counts())
print(df["Machine failure"].value_counts(normalize=True).round(4))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

### Observation: Data Overview

The dataset has 10,000 rows and 14 columns with no missing values. Only 339 rows (3.39%) are failures, so the data is severely imbalanced: a model that always predicted “no failure” would reach 96.6% accuracy while detecting nothing. Accuracy is therefore not a useful metric, and I will focus on precision, False Discovery Rate and recall. The columns TWF, HDF, PWF, OSF and RNF describe the failure type and directly determine Machine failure, so they will not be used as features.

## Data cleaning and preparation

In [5]:
# Rename columns (XGBoost does not accept [ ] in names)
df = df.rename(columns={
    "Air temperature [K]": "Air_temp",
    "Process temperature [K]": "Process_temp",
    "Rotational speed [rpm]": "Rot_speed",
    "Torque [Nm]": "Torque",
    "Tool wear [min]": "Tool_wear",
    "Machine failure": "Machine_failure",
})

# Drop ID columns (they only identify rows)
df = df.drop(columns=["UDI", "Product ID"])

# Encode product quality: L < M < H
df["Type"] = df["Type"].map({"L": 0, "M": 1, "H": 2})

# Check that Machine_failure matches the failure-type columns
failure_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
mismatch = df[(df["Machine_failure"] == 1) & (df[failure_cols].sum(axis=1) == 0)]
print("Failures with no failure type flagged:", len(mismatch))
print(df.isnull().sum().sum(), "missing values |", df.duplicated().sum(), "duplicate rows")
df.head()

Failures with no failure type flagged: 9
0 missing values | 0 duplicate rows


,Type,Air_temp,Process_temp,Rot_speed,Torque,Tool_wear,Machine_failure,TWF,HDF,PWF,OSF,RNF
0,1,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,0,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,0,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,0,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,0,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


### Observation: Data Cleaning

The dataset has no missing values and no duplicate rows. The ID columns (UDI, Product ID) were dropped because they only identify rows, the column names were simplified (XGBoost does not accept brackets in names), and the product quality Type was encoded as 0, 1 and 2 (Low, Medium, High). The failure-type columns (TWF, HDF, PWF, OSF, RNF) will not be used as features because they define Machine_failure directly. Nine failure rows have no failure type flagged, which contradicts the dataset description, but I kept them because they are labeled failures and are only 9 of 339.

## Exploratory Data Analysis(EDA)